# 06 — Tải audio gốc cho TRAIN context
Chế độ hiện tại: tải toàn bộ audio của danh sách TRAIN trong giới hạn dung lượng. Bạn tự chạy cell cấu hình → kiểm tra → tải. Notebook không huấn luyện và không dùng GPU. File FLAC hợp lệ đã có sẽ được bỏ qua; có thể chạy tiếp sau khi ngắt.

Ngân sách mặc định: thư mục audio 10 GB (tính cả file tạm); giữ ổ chứa audio còn ít nhất 20 GB trống. Bộ giám sát kiểm tra mỗi giây và có phần dự phòng, không phải quota cứng của hệ điều hành. Video trên 60 phút được bỏ qua trong lô đầu. Không tải hình/video để lưu, không xóa dataset hay checkpoint.

Nguồn công cụ: [yt-dlp](https://github.com/yt-dlp/yt-dlp), [EJS/Deno](https://github.com/yt-dlp/yt-dlp/wiki/EJS), [Deno Windows release](https://github.com/denoland/deno/releases). Chỉ các cell cài công cụ bên dưới mới tải phần mềm khi bạn bấm chạy.

In [1]:
from pathlib import Path
import sys, json
here = Path.cwd().resolve()
PROJECT = next((p for p in [here, *here.parents] if (p/'asr/config.py').is_file()), None)
if PROJECT is None:
    PROJECT = next((p/'ASR' for p in [here, *here.parents] if (p/'ASR/asr/config.py').is_file()), None)
assert PROJECT is not None, 'Mở notebook trong workspace DPL'
PYTHON = PROJECT.parent/'.venv-asr/Scripts/python.exe'
assert PYTHON.is_file(), 'Chưa có môi trường .venv-asr'
sys.path.insert(0, str(PROJECT/'local'))
from notebook_process import run_cli as _run_cli
def run_cli(arguments):
    return _run_cli(PROJECT, PYTHON, arguments)
DOWNLOADER = PROJECT/'scripts/download_context_audio.py'
DOWNLOAD_ALL = True
BATCH_VIDEOS = 30  # Only used when DOWNLOAD_ALL=False
PAUSE_SECONDS = 8
AUDIO_BUDGET_GB = 10
MIN_FREE_GB = 20
MAX_VIDEO_MINUTES = 60
RETRY_FAILED = True  # Retry the videos that failed before cookie authentication worked
COOKIE_BROWSER = None  # Chrome direct access failed with DPAPI; use local file
COOKIE_FILE = PROJECT/'local/secrets/youtube.cookies.txt'
AUDIO_DIR = PROJECT/'local/imports/original_audio/train'
options = ['--batch-videos', str(BATCH_VIDEOS), '--audio-budget-gb', str(AUDIO_BUDGET_GB),
           '--min-free-gb', str(MIN_FREE_GB), '--max-video-minutes', str(MAX_VIDEO_MINUTES)]
options += ['--pause-seconds', str(PAUSE_SECONDS)]
if DOWNLOAD_ALL:
    options += ['--all-videos']
if COOKIE_FILE:
    options += ['--cookies', str(Path(COOKIE_FILE).expanduser().resolve())]
elif COOKIE_BROWSER:
    options += ['--cookies-from-browser', COOKIE_BROWSER]
print('Nguồn:', PROJECT/'local/context/train_candidates.csv')
print('Audio lưu tại:', AUDIO_DIR)
print('Chế độ:', 'TOÀN BỘ DANH SÁCH' if DOWNLOAD_ALL else f'Lô {BATCH_VIDEOS}', '| Ngân sách audio:', AUDIO_BUDGET_GB, 'GB | Giữ trống:', MIN_FREE_GB, 'GB')


Nguồn: D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\context\train_candidates.csv
Audio lưu tại: D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\imports\original_audio\train
Chế độ: TOÀN BỘ DANH SÁCH | Ngân sách audio: 10 GB | Giữ trống: 20 GB


## 1. Cài công cụ tải (chạy lần đầu)
Cell này chỉ cài/cập nhật yt-dlp và gói chứa FFmpeg vào .venv-asr. Không cài lại PyTorch/Transformers, không khởi động model. Nếu đã cài thành công có thể bỏ qua ở những lô sau.

In [2]:
run_cli(['-m', 'pip', 'install', '--upgrade', 'yt-dlp[default]', 'imageio-ffmpeg'])

[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u -m pip install --upgrade yt-dlp[default] imageio-ffmpeg
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261020_022606_060983.log
[notice] A new release of pip available: 22.2.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
[END] exit=0 | elapsed=5.0s | log=D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261020_022606_060983.log


## 2. Chuẩn bị Deno cho YouTube
Dùng Deno trên PATH nếu đã có. Nếu chưa có, cell tải bản portable chính thức vào ASR/local/tools/deno; không cài toàn hệ thống, không cần sửa PATH. File đã có sẽ được giữ lại.

In [3]:
run_cli([DOWNLOADER, 'install-deno'])

[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\download_context_audio.py" install-deno
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261020_022611_125489.log
Deno already present: D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\tools\deno\deno.exe
[END] exit=0 | elapsed=0.2s | log=D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261020_022611_125489.log


## 3. Kiểm tra trước tải
In số clip/video, dung lượng đang dùng, dung lượng ổ trống và công cụ. Chưa tải audio ở cell này.

In [4]:
run_cli([DOWNLOADER, 'check', *options])

[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\download_context_audio.py" check --batch-videos 30 --audio-budget-gb 10 --min-free-gb 20 --max-video-minutes 60 --pause-seconds 8 --all-videos --cookies "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\secrets\youtube.cookies.txt"
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261020_022611_401329.log
{
  "candidate_clips": 3899,
  "candidate_videos": 1276,
  "audio_folder": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\ASR\\local\\imports\\original_audio\\train",
  "audio_folder_gb": 0.002789152,
  "drive_free_gb": 119.891329024,
  "audio_budget_gb": 10.0,
  "min_free_gb": 20.0,
  "yt_dlp": "2026.8.19",
  "deno": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\ASR\\local\\tools\\deno\\deno.exe",
  "ffmpeg": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\lib\\site-packages\\imageio_ffmpeg\\binaries\\ffmpeg-win-x86_64-v7.1.exe",
  "authentication": "local cookie

## 4. Tải toàn bộ audio TRAIN bằng file cookie cục bộ
Log đã xác nhận lỗi Failed to decrypt with DPAPI: không tiếp tục đọc cookie trực tiếp từ Chrome. Notebook hiện dùng COOKIE_FILE = PROJECT/'local/secrets/youtube.cookies.txt', COOKIE_BROWSER=None.

Trên Chrome, dùng công cụ xuất cookie Netscape được [tài liệu yt-dlp liên kết](https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp), ví dụ Get cookies.txt LOCALLY. Mở YouTube bằng phiên bạn đang xem được, chỉ xuất cookie của trang YouTube hiện tại (không xuất tất cả website). Lưu tên youtube.cookies.txt vào thư mục ASR/local/secrets đã chuẩn bị. Không dán nội dung vào notebook/chat và không commit file; thư mục đã được bỏ qua trong Git. Notebook không đọc hay in nội dung cookie vào output.

Một video đã tải thành công. DOWNLOAD_ALL=True hiện duyệt toàn bộ danh sách (1.276 video tại thời điểm lập manifest), bỏ qua FLAC đã có và thử lại các lỗi cũ bằng cookie. Nghỉ 8 giây giữa hai video. Nếu HTTP 429/bot-check hoặc lỗi cookie xuất hiện thì dừng lô ngay; có cookie không bảo đảm hết chặn.
Bấm ▶ cell dưới là bắt đầu tải. Tải tuần tự một video/lần, giới hạn tốc độ 2 MiB/s; chuyển sang FLAC mono 16 kHz trên CPU. File gốc tải tạm được xóa sau khi FLAC đã kiểm tra và lưu thành công. File tạm của lượt lỗi được giữ để tiếp tục. Chỉ giữ một FLAC/video.

Muốn tải lô tiếp, chạy lại cell này sau khi kiểm tra kết quả lô đầu. Nếu muốn thử lại video lỗi, đặt RETRY_FAILED=True rồi chạy lại cell tải. Không chạy đồng thời hai lượt tải. Interrupt dừng cả tiến trình con; tiến độ đã ghi trên ổ đĩa được giữ.

Các clip có end_s vượt thời lượng audio không được đưa vào danh sách có audio. Danh sách candidate gốc và nhãn train/dev không bị sửa.

In [5]:
run_cli([DOWNLOADER, 'download', *options, *(['--retry-failed'] if RETRY_FAILED else [])])

[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\download_context_audio.py" download --batch-videos 30 --audio-budget-gb 10 --min-free-gb 20 --max-video-minutes 60 --pause-seconds 8 --all-videos --cookies "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\secrets\youtube.cookies.txt" --retry-failed
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261020_022611_861593.log
{
  "candidate_clips": 3899,
  "candidate_videos": 1276,
  "audio_folder": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\ASR\\local\\imports\\original_audio\\train",
  "audio_folder_gb": 0.002789152,
  "drive_free_gb": 119.891324928,
  "audio_budget_gb": 10.0,
  "min_free_gb": 20.0,
  "yt_dlp": "2026.8.19",
  "deno": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\ASR\\local\\tools\\deno\\deno.exe",
  "ffmpeg": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\lib\\site-packages\\imageio_ffmpeg\\binaries\\ffmpeg-win-x86_64-v7.1.exe",
  "authenticati

## 5. Xem kết quả lô vừa tải
Log lịch sử: ASR/local/context/train_audio_downloads.jsonl. Kết quả lô mới nhất: train_audio_last_batch.csv. Clip có audio và đủ thời lượng: train_audio_available.csv. alignment_status vẫn pending: chưa xác nhận audio video khớp dataset. Chưa dùng ngay để train gate.

In [6]:
import pandas as pd
from IPython.display import display
context = PROJECT/'local/context'
last_batch = context/'train_audio_last_batch.csv'
if last_batch.is_file():
    batch = pd.read_csv(last_batch)
    display(batch[['video_id', 'status', 'duration_s', 'size_mb', 'error']])
ready_path = context/'train_audio_available.csv'
if ready_path.is_file():
    ready = pd.read_csv(ready_path)
    print('Clip có audio:', len(ready), '| Video:', ready['video_id'].nunique())
    display(ready.head(10))
else:
    print('Chưa có danh sách audio. Chạy cell tải hoặc xem lỗi trước đó.')
run_cli([DOWNLOADER, 'check', *options])


,video_id,status,duration_s,size_mb,error
0,-8Cnu_e9liw,ok,228.217312,3.103636,NaN
1,-BFovvWaBXc,ok,349.646063,6.817200,NaN
2,-D0P6NEowYU,ok,776.080563,13.074644,NaN
3,-F_vV4tQW-U,ok,925.036562,11.950722,NaN
4,-GxLwh-ck2A,ok,220.252875,3.077746,NaN
...,...,...,...,...,...
1270,ztaB9O5pmPk,ok,723.057812,13.971321,NaN
1271,zuxUdd-W2gc,ok,620.576500,13.571533,NaN
1272,zw_nCaTdqrI,ok,143.487750,2.422505,NaN
1273,zxqcgz525jQ,ok,157.071375,3.645858,NaN


Clip có audio: 3459 | Video: 1145


,segment_id,video_id,audio_path,history_start_s,start_s,end_s,alignment_status
0,Med_CS-3161-3,-2TKFvk-qFA,D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\...,76.0,96.0,105.0,pending
1,Med_CS-3161-4,-2TKFvk-qFA,D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\...,86.0,106.0,116.0,pending
2,Med_CS-2800-3,-8Cnu_e9liw,D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\...,83.0,103.0,110.0,pending
3,Med_CS-444-2,-BFovvWaBXc,D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\...,27.0,47.0,58.0,pending
4,Med_CS-444-3,-BFovvWaBXc,D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\...,39.0,59.0,69.0,pending
5,Med_CS-444-6,-BFovvWaBXc,D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\...,80.0,100.0,113.0,pending
6,Med_CS-1274_1-8,-D0P6NEowYU,D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\...,31.0,51.0,64.0,pending
7,Med_CS-1274_2-38,-D0P6NEowYU,D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\...,320.0,340.0,350.0,pending
8,Med_CS-1274_2-39,-D0P6NEowYU,D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\...,331.0,351.0,372.0,pending
9,Med_CS-5103_2-3,-F_vV4tQW-U,D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\...,64.0,84.0,90.0,pending


[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\download_context_audio.py" check --batch-videos 30 --audio-budget-gb 10 --min-free-gb 20 --max-video-minutes 60 --pause-seconds 8 --all-videos --cookies "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\secrets\youtube.cookies.txt"
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261020_092614_686326.log
{
  "candidate_clips": 3899,
  "candidate_videos": 1276,
  "audio_folder": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\ASR\\local\\imports\\original_audio\\train",
  "audio_folder_gb": 9.18851925,
  "drive_free_gb": 110.669135872,
  "audio_budget_gb": 10.0,
  "min_free_gb": 20.0,
  "yt_dlp": "2026.8.19",
  "deno": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\ASR\\local\\tools\\deno\\deno.exe",
  "ffmpeg": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\lib\\site-packages\\imageio_ffmpeg\\binaries\\ffmpeg-win-x86_64-v7.1.exe",
  "authentication": "local cookie 